# Task 1 · Minimal pipeline

Iteration 1 · Exploratory data analysis (EDA). Notebook documenting the task as required by `AGENTS.md` §4.

## 1. Prompt

The person first asked:

> can we do a run?

The agent ran `uv run python -m src.pipeline`, which failed with `No module named src.pipeline`, and proposed doing task 1:
a new `src/pipeline.py`, a unit test in `tests/unit/test_pipeline.py`, this notebook and the update of `TASKS.md`,
keeping the original column names because their translation is an open question (SPECS.md §8). The person answered:

> do it

## 2. Task and scope

**Task (TASKS.md, Iteration 1):**

> 1. Minimal pipeline: `uv run python -m src.pipeline` reads the CSV with explicit types and writes `data/processed/accidents.parquet` without cleaning (SPECS §4; R1, R2; CA5, CA6, CA7)

**Requirements covered (SPECS.md §5):**

- **R1.** The pipeline reads `data/raw/accidents_bcn_2019_2025.csv` and writes every row and column to `data/processed/accidents.parquet` without changing any value.
- **R2.** The pipeline reads the CSV with explicit types: `fecha` as a date; counts, codes and date parts as integers (nullable where values are missing); `longitud` and `latitud` as decimals; `es_causa_vianant` and `accidente_con_victimas` as booleans.

**Acceptance criteria (SPECS.md §6, `tests/acceptance/test_data.py`):** CA5 (same rows and columns as the CSV), CA6 (same values) and CA7 (column types).

**Out of scope:** any cleaning, renaming or translating columns or values (open question in SPECS.md §8), the exploration functions (tasks 2–6) and everything in `dashboard/` (R10–R13).

## 3. Principles and best practices applied

| Principle | Where | Why |
| --- | --- | --- |
| Single responsibility, small functions | `src/pipeline.py`: `read_accidents` only reads and types, `write_parquet` only writes, `run` only connects them | Each step can be tested on its own (`AGENTS.md` §6) |
| No hidden side effects | `run(raw_csv, output)` receives its paths as parameters, with the project paths as defaults | The unit test runs the whole pipeline in a temporary folder without touching `data/processed/` |
| Explicit configuration instead of inference | `COLUMN_TYPES` lists all 28 non-date columns in CSV order; `fecha` is parsed with an explicit `DATE_FORMAT` | R2 asks for explicit types; pandas' inference would read `codi_districte` as float because of its missing values |
| Parquet for intermediate tables | `write_parquet` | `AGENTS.md` §8 and `ARCHITECTURE.md` §2 (`data/processed/` holds Parquet files) |
| No hard-coded results | The acceptance tests compare against the CSV read in the test itself; the unit test uses a hand-written two-row CSV | `AGENTS.md` §5 |
| Docstrings cite the requirement | Every function in `src/pipeline.py` | `AGENTS.md` §6 |
| Logic in `src/`, notebook only calls it | Section 6 of this notebook imports `src.pipeline` | `ARCHITECTURE.md` §2 (`notebooks/` import from `src/`) |
| Minimum implementation | No cleaning, no renaming, no extra outputs | `AGENTS.md` §3.2 and the task's "without cleaning" |

## 4. Reasoning and decisions

- **Typing while reading vs. converting afterwards.** `pd.read_csv(dtype=..., parse_dates=..., date_format=...)` sets every type in one step; converting after reading would first let pandas infer types (e.g. `codi_districte` as float) and then convert them. Chosen: typing while reading.
- **Which integer type.** `codi_districte` has 232 missing values in the raw CSV, so it uses pandas' nullable `Int64`; the other integer columns have no missing values and use `int64`. Using `Int64` everywhere was also possible, but R2 asks for nullable integers only where values are missing.
- **Booleans.** `es_causa_vianant` and `accidente_con_victimas` are stored as `TRUE`/`FALSE` in the CSV, which pandas reads directly with `dtype="bool"`; no mapping was needed.
- **Text columns.** Declared as `"str"`, pandas 3's default string type, so they are also explicit.
- **Column names.** Kept as in the CSV (Catalan and Spanish). *Assumption*: the agent stated this default before starting and the person answered "do it"; the translation itself is still an open question in SPECS.md §8.
- **Notebook execution.** `nbclient`/`nbconvert` are not project dependencies. Instead of adding one, the notebook was executed with `jupyter_client` and `ipykernel` (already installed) by a helper script outside the repository.

The implementation passed its tests at the first run; no attempts were reverted.

## 5. Changes to the source code

| archivo | tipo de cambio | motivo | R/CA |
| --- | --- | --- | --- |
| `src/pipeline.py` | created | Reads the raw CSV with explicit types and writes `accidents.parquet` | R1, R2 / CA5, CA6, CA7 |
| `tests/unit/test_pipeline.py` | created | Unit tests of the typing and of the whole run on a hand-written CSV | R1, R2 |
| `notebooks/bitacora/2026-11-26_T1_minimal_pipeline.ipynb` | created | This notebook (`AGENTS.md` §4) | — |
| `TASKS.md` | modified | Task 1 marked `[x]` with the `Agente:` line | — |

The acceptance tests (`tests/acceptance/test_data.py`) were written before this task, at the person's request, and were not changed.

**Excerpt of `src/pipeline.py`** (the `COLUMN_TYPES` dictionary lists all 28 non-date columns; abbreviated here):

```python
DATE_COLUMNS = ["fecha"]
DATE_FORMAT = "%Y-%m-%d"

COLUMN_TYPES: dict[str, str] = {
    "numero_expedient": "str",
    "codi_districte": "Int64",   # nullable: 232 missing values
    "nk_any": "int64",
    # ...
    "longitud": "float64",
    "es_causa_vianant": "bool",
    # ...
}


def read_accidents(path: Path) -> pd.DataFrame:
    """Reads the raw accidents CSV with an explicit type for every column (R2)."""
    return pd.read_csv(path, dtype=COLUMN_TYPES, parse_dates=DATE_COLUMNS, date_format=DATE_FORMAT)


def write_parquet(table: pd.DataFrame, path: Path) -> None:
    """Writes a table to Parquet without changing any row, column or value (R1)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_parquet(path, index=False)


def run(raw_csv: Path = RAW_CSV, output: Path = ACCIDENTS_PARQUET) -> None:
    """Writes the raw accidents CSV to accidents.parquet with explicit types and no cleaning (R1, R2)."""
    write_parquet(read_accidents(raw_csv), output)
```

## 6. Verification

Commands run from the repository root, with their actual output.

**`uv run python -m src.pipeline`**: finished with exit code 0 and wrote `data/processed/accidents.parquet` (1,698,433 bytes).

**`uv run pytest tests/unit tests/acceptance/test_data.py -q`** (tests applicable to this task):

```
.....................                                                    [100%]
21 passed in 2.89s
```

**`uv run pytest -q`** (whole suite). The 9 errors belong to later tasks: the exploration tests need `src/exploration.py` (tasks 2–6) and the dashboard tests need `dashboard/` (Iteration 6).

```
ERROR tests/acceptance/test_dashboard.py::test_dashboard_files_regenerated - ...
ERROR tests/acceptance/test_dashboard.py::test_charts_load_data_by_url - Fail...
ERROR tests/acceptance/test_dashboard.py::test_json_files_are_aggregated - Fa...
ERROR tests/acceptance/test_exploration.py::test_profile_fields - ModuleNotFo...
ERROR tests/acceptance/test_exploration.py::test_check_derived_columns - Modu...
ERROR tests/acceptance/test_exploration.py::test_check_identity_and_coverage
ERROR tests/acceptance/test_exploration.py::test_check_geography - ModuleNotF...
ERROR tests/acceptance/test_exploration.py::test_count_by - ModuleNotFoundErr...
ERROR tests/acceptance/test_exploration.py::test_describe_counts - ModuleNotF...
21 passed, 9 errors in 2.52s
```

**`uv run python scripts/traceability.py`**: exit code 1, because CA3 and CA4 have no test yet (see section 7). The criteria of this task are covered.

```
✔ CA1: tests\acceptance\test_dashboard.py
✔ CA2: tests\acceptance\test_dashboard.py
✘ CA3: NO TEST
✘ CA4: NO TEST
✔ CA5: tests\acceptance\test_data.py
✔ CA6: tests\acceptance\test_data.py
✔ CA7: tests\acceptance\test_data.py
✔ CA8: tests\acceptance\test_exploration.py
✔ CA9: tests\acceptance\test_exploration.py
✔ CA10: tests\acceptance\test_exploration.py
✔ CA11: tests\acceptance\test_exploration.py
✔ CA12: tests\acceptance\test_exploration.py

2 criterion/criteria without a test and 0 unknown reference(s).
```

**Demonstration on the real data.** The cells below run the pipeline from `src/` and compare its output with the raw CSV.

In [1]:
import sys
from pathlib import Path

# Make the repository root importable, wherever the notebook is opened from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.pipeline import ACCIDENTS_PARQUET, RAW_CSV, run

In [2]:
run()
accidents = pd.read_parquet(ACCIDENTS_PARQUET)
raw = pd.read_csv(RAW_CSV, low_memory=False)

print("Written:", ACCIDENTS_PARQUET.relative_to(ROOT).as_posix())
print("Rows    (parquet, csv):", len(accidents), len(raw))
print("Columns (parquet, csv):", accidents.shape[1], raw.shape[1])
print("Same column order:", list(accidents.columns) == list(raw.columns))

Written: data/processed/accidents.parquet
Rows    (parquet, csv): 54954 54954
Columns (parquet, csv): 29 29
Same column order: True


In [3]:
# R2: type of each column, compared with what pandas infers from the CSV without explicit types.
pd.DataFrame({"parquet": accidents.dtypes.astype(str), "csv_inferred": raw.dtypes.astype(str)})

,parquet,csv_inferred
numero_expedient,str,str
codi_districte,Int64,float64
nom_districte,str,str
nom_barri,str,str
nom_carrer,str,str
nk_any,int64,int64
mes_any,int64,int64
nom_mes,str,str
dia_mes,int64,int64
hora_dia,int64,int64


In [4]:
# R1 / CA6: same expedients, same total victims and same missing values per column.
print("Same numero_expedient values:", sorted(accidents["numero_expedient"]) == sorted(raw["numero_expedient"]))
print("Total numero_victimes (parquet, csv):", accidents["numero_victimes"].sum(), raw["numero_victimes"].sum())
missing = pd.DataFrame({"parquet": accidents.isna().sum(), "csv": raw.isna().sum()})
missing[missing.any(axis=1)]

Same numero_expedient values: True
Total numero_victimes (parquet, csv): 63007 63007


,parquet,csv
codi_districte,232,232
nom_districte,232,232
nom_barri,232,232
nom_carrer,1,1
longitud,42,42
latitud,42,42


In [5]:
accidents.head(3)

,numero_expedient,codi_districte,nom_districte,nom_barri,nom_carrer,nk_any,mes_any,nom_mes,dia_mes,hora_dia,...,latitud,fichero_origen,es_causa_vianant,fecha,dia_semana,fin_de_semana,franja_horaria,total_lesionados,gravedad,accidente_con_victimas
0,2019S000001,5,Sarrià-Sant Gervasi,Sant Gervasi - Galvany,Diagonal / Augusta,2019,1,Gener,1,1,...,41.395744,2019_accidents_gu_bcn.csv,False,2019-01-01,Tuesday,Laborable,Madrugada (00-06h),1,Grave,True
1,2019S000002,9,Sant Andreu,el Congrés i els Indians,Felip II / Congrés Eucarístic,2019,1,Gener,1,4,...,41.425361,2019_accidents_gu_bcn.csv,False,2019-01-01,Tuesday,Laborable,Madrugada (00-06h),1,Leve,True
2,2019S000003,10,Sant Martí,el Parc i la Llacuna del Poblenou,Pallars,2019,1,Gener,1,5,...,41.396887,2019_accidents_gu_bcn.csv,True,2019-01-01,Tuesday,Laborable,Madrugada (00-06h),1,Leve,True


## 7. Pending issues

**Tests deferred to later tasks (not failures of this task):**

- `tests/acceptance/test_exploration.py` (CA8–CA12) needs `src/exploration.py`, built in tasks 2–6.
- `tests/acceptance/test_dashboard.py` (CA1, CA2) needs `dashboard/`, built in Iteration 6.
- CA3 and CA4 have no test: they need the date format and the data-source presentation in `DESIGN.md` (empty "Dates" row) and the name of the summary JSON table. Until then `scripts/traceability.py` exits with code 1.

**Noticed outside the scope of this task (reported, not fixed):**

- `descripcio_causa_vianant` spells the same value in two ways: `No és causa del vianant` and `No es causa del vianant` (see task 3 and R4).
- The raw CSV has 54,954 rows, while `AGENTS.md` §8 and `ARCHITECTURE.md` §5 mention almost two million (open question in SPECS.md §8).
- `ARCHITECTURE.md` §1, §3 and §4 are still empty; the `pipeline.py` row of the modules table could now be filled in by the team.
- `SPECS.md` §3 still says "N rows"; recording the row count is part of task 2.